# DDPM on MNIST — Colab T4 Training

Before running:
1. **Runtime → Change runtime type → T4 GPU** (Hardware accelerator)
2. Make sure `unet.py`, `ddpm.py`, `train_mnist.py` are in the same Google Drive folder as this notebook (default expected path: `MyDrive/diffusion_mnist/`).

In [ ]:
# ---- 1. verify T4 GPU is attached ----
!nvidia-smi

In [ ]:
# ---- 2. mount Google Drive and cd into project dir ----
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/diffusion_mnist'  # adjust if you uploaded elsewhere
os.chdir(PROJECT_DIR)
print('cwd =', os.getcwd())
!ls -la

In [ ]:
# ---- 3. quick sanity: model + diffusion forward ----
import sys; sys.path.insert(0, '.')
import torch
from unet import UNet, count_parameters
from ddpm import DDPM

model = UNet(num_classes=None).cuda()
print('params:', f'{count_parameters(model):,}')
diff = DDPM(num_timesteps=1000).cuda()
x = torch.randn(4, 1, 28, 28, device='cuda')
t = torch.randint(0, 1000, (4,), device='cuda')
with torch.amp.autocast('cuda'):
    out = model(x, t)
print('forward OK:', tuple(out.shape))

## Stage A — Train unconditional DDPM (≈50k steps, ~1.5 h on T4)

In [ ]:
# Outputs (checkpoints + sample grids + loss.png) go to Drive so survive disconnects.
# To resume after disconnect, add:  --resume ./runs/mnist_uncond/last.pt

!python train_mnist.py \
    --steps 50000 \
    --batch-size 128 \
    --lr 2e-4 \
    --num-timesteps 1000 \
    --sample-every 5000 \
    --ckpt-every 5000 \
    --num-workers 2 \
    --out-dir ./runs/mnist_uncond

In [ ]:
# show the latest sample grid and the loss curve
import glob, os
from IPython.display import Image, display

samples = sorted(glob.glob('./runs/mnist_uncond/samples/*.png'))
if samples:
    print(samples[-1])
    display(Image(samples[-1]))
if os.path.exists('./runs/mnist_uncond/loss.png'):
    display(Image('./runs/mnist_uncond/loss.png'))

## Stage B — Train class-conditional DDPM with Classifier-Free Guidance (≈50k steps, ~1.5 h)

In [ ]:
!python train_mnist.py \
    --steps 50000 \
    --batch-size 128 \
    --lr 2e-4 \
    --num-timesteps 1000 \
    --conditional \
    --cfg-dropout 0.1 \
    --cfg-scale 3.0 \
    --sample-every 5000 \
    --ckpt-every 5000 \
    --num-workers 2 \
    --out-dir ./runs/mnist_cond

In [ ]:
samples = sorted(glob.glob('./runs/mnist_cond/samples/*.png'))
if samples:
    print(samples[-1])
    display(Image(samples[-1]))
if os.path.exists('./runs/mnist_cond/loss.png'):
    display(Image('./runs/mnist_cond/loss.png'))

## Download final checkpoints back to local

The `final_ema.pt` files inside `runs/mnist_uncond/` and `runs/mnist_cond/` are already in Drive — just download them from the Drive web UI to `project/checkpoints/` on your local machine.

Alternatively, use the cell below to download via the Colab files panel.

In [ ]:
from google.colab import files
for p in ['./runs/mnist_uncond/final_ema.pt', './runs/mnist_cond/final_ema.pt']:
    if os.path.exists(p):
        files.download(p)